**[Source]** Integrated Project
**[Status]** NEW
**[Role]** 지수 전처리 결과(data/preprocessed_final) 독립 검증 — 결과가 PASS여야 모델링(06~)으로 진행
**[Modification]** 데이터는 수정하지 않고 읽기만 한다(검증만). 지수 파일을 다시 가공·재분할·삭제하지 않는다.
**Test 사용 방식:** Test 파일은 행 수·문서 누수·중복·길이 통계 같은 *구조 검증*에만 읽는다. 성능 평가·모델/규칙 선택에는 쓰지 않으며, 예시 문장 출력은 Train/Validation에서만 한다.

# 05. 지수 전처리 결과 독립 검증

**이 노트북이 하는 일:** 지수 FIXED 노트북(01~04)이 만든 `data/preprocessed_final`이 실제로 정상인지 독립적으로 다시 확인하고 **PASS / STOP**을 판정한다.

**이 노트북이 하지 않는 일:** 데이터를 다시 만들거나, 새 split을 만들거나, 문제 행을 삭제하는 것. 문제가 발견되어도 파일을 자동 수정하지 않고 보고만 한다.

## 판정 기준 (결과를 보기 전에 미리 고정)

| 등급 | 항목 | 기준 |
|---|---|---|
| STOP | 파일 SHA256 | FINAL_MANIFEST.json과 불일치 |
| STOP | 행 수 | manifest의 counts_final과 불일치 |
| STOP | 필수 column | `document_id`, `utterance_id`, `input`, `target`, `split` 중 누락 |
| STOP | 결측·빈 문자열 | input 또는 target이 null/비문자열/공백뿐인 행 ≥ 1 |
| STOP | document 누수 | split 간 `document_id` 교집합 ≥ 1 |
| STOP | utterance_id | split 내 중복 또는 split 간 겹침 ≥ 1 |
| STOP | split 표기 | 파일별 `split` 필드가 파일 이름과 다른 행 ≥ 1, 또는 document_split_map.csv와 배정이 다른 문서 ≥ 1 |
| STOP | 근접중복 그룹 설계 위반 | 긴 문장(공백 제외 15자 이상, 한글 5자 이상) 입력이 Train과 Validation/Test에 동시에 존재하는 행이 해당 split 전체의 **0.1% 초과** (지수 전처리는 이런 문장을 공유하는 문서를 같은 split에 묶도록 설계됨) |
| WARN | 짧은 문장/일반 문장의 split 간 동일 input·pair | 정상적인 짧은 대화(“ㅋㅋ”, “네”)가 겹칠 수 있으므로 삭제하지 않고 **비율만 보고** |
| WARN | 길이·내용 보존 후보 | 자동 삭제하지 않고 후보 수와 예시만 출력 |

WARN은 모델링을 막지 않는다. 다만 결과 해석(특히 Test 점수)에 영향을 줄 수 있으므로 최종 보고서에 그대로 기록한다.

In [1]:
# [셀 1] 환경, 경로, 재현성 기록
import os, sys, json, re, time, random, hashlib, platform, unicodedata
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)

def find_root():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "config" / "paths.json").exists():
            return c
    raise FileNotFoundError("config/paths.json이 있는 통합 프로젝트 루트를 찾지 못했습니다. notebooks 폴더에서 실행하세요.")

ROOT = find_root()
CFG_PATHS = json.loads((ROOT / "config" / "paths.json").read_text(encoding="utf-8"))
JISOO_ROOT = (ROOT / CFG_PATHS["jisoo_root"]).resolve()
DATA_DIR = JISOO_ROOT / CFG_PATHS["jisoo_data_subdir"]
OUT_DIR = ROOT / "data" / "processed" / "verification_05"; OUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {s: DATA_DIR / f"{s}.jsonl" for s in ("train", "validation", "test")}
EXTRA = {"manifest": DATA_DIR / "FINAL_MANIFEST.json", "split_map": DATA_DIR / "document_split_map.csv",
         "summary": DATA_DIR / "final_summary.csv", "config": DATA_DIR / "preprocess_config.json",
         "removed": DATA_DIR / "removed_rows.jsonl", "frozen": DATA_DIR / "FROZEN.txt"}
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__, "|", platform.platform())
print("통합 프로젝트 루트:", ROOT); print("지수 최종 데이터:", DATA_DIR)
for k, p in {**FILES, **EXTRA}.items():
    print(f"  {k:11s} 존재={p.exists()}  {p.stat().st_size if p.exists() else 0:>14,d} bytes")
assert all(p.exists() for p in {**FILES, **EXTRA}.values()), "필수 파일이 없습니다 → STOP"

CHECKS = []   # (등급, 항목, 통과 여부, 상세)
def check(level, name, ok, detail=""):
    CHECKS.append({"level": level, "check": name, "pass": bool(ok), "detail": str(detail)})
    print(("통과 " if ok else ("실패 " if level == "STOP" else "주의 ")) + f"[{level}] {name}" + (f" — {detail}" if detail else ""))

Python 3.10.12 | pandas 2.3.3 | numpy 2.2.6 | Linux-6.8.0-138-generic-x86_64-with-glibc2.35
통합 프로젝트 루트: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/integrated_korean_correction
지수 최종 데이터: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/data/preprocessed_final
  train       존재=True   1,096,704,146 bytes
  validation  존재=True      61,771,030 bytes
  test        존재=True      62,034,177 bytes
  manifest    존재=True           2,060 bytes
  split_map   존재=True       1,250,818 bytes
  summary     존재=True             222 bytes
  config      존재=True           1,220 bytes
  removed     존재=True           3,412 bytes
  frozen      존재=True             132 bytes


## [셀 2] 동결 기록 확인 (SHA256, manifest)
지수 `FINAL_MANIFEST.json`에 기록된 SHA256과 실제 파일의 SHA256을 다시 계산해 비교한다. 이 값이 다르면 동결 이후 파일이 바뀐 것이다.

In [2]:
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for ch in iter(lambda: f.read(1 << 22), b""):
            h.update(ch)
    return h.hexdigest()

MANIFEST = json.loads(EXTRA["manifest"].read_text(encoding="utf-8"))
print("dataset_version:", MANIFEST["dataset_version"], "| frozen:", MANIFEST.get("frozen"), "| created_at:", MANIFEST["created_at"])
print("FROZEN.txt:", EXTRA["frozen"].read_text(encoding="utf-8").strip().replace("\n", " / "))
t0 = time.time(); ACTUAL_SHA = {}
for s, p in FILES.items():
    ACTUAL_SHA[s] = sha256(p)
    expected = MANIFEST["sha256"][f"{s}.jsonl"]
    check("STOP", f"SHA256 {s}.jsonl = manifest", ACTUAL_SHA[s] == expected, ACTUAL_SHA[s][:16] + "…")
print(f"SHA256 계산 {time.time()-t0:.0f}초")
src = MANIFEST.get("sha256_v3b_source", {})
print("v3b(제거 전) 대비 validation/test 해시 동일:", {s: src.get(f"{s}.jsonl") == MANIFEST["sha256"][f"{s}.jsonl"] for s in ("validation", "test")}, "(train은 수동 판정 T 10행 제거로 달라지는 것이 정상)")

dataset_version: preprocessed_final_v1 | frozen: True | created_at: 2026-09-20T17:28:30+09:00
FROZEN.txt: preprocessed_final_v1 frozen at 2026-09-20T17:28:30+09:00 / 수정 금지. 무결성은 FINAL_MANIFEST.json의 sha256으로 확인.
통과 [STOP] SHA256 train.jsonl = manifest — 84181f125239fa30…
통과 [STOP] SHA256 validation.jsonl = manifest — c1dd6c2a174cb5f1…
통과 [STOP] SHA256 test.jsonl = manifest — 95dd6c6be6ff83a1…
SHA256 계산 5초
v3b(제거 전) 대비 validation/test 해시 동일: {'validation': True, 'test': True} (train은 수동 판정 T 10행 제거로 달라지는 것이 정상)


## [셀 3] 스트리밍 로드 (읽기 전용)
1.1GB Train 파일을 한 줄씩 읽어 필요한 필드만 보관한다(메모리 절약). 데이터는 어디에도 다시 저장하지 않는다.

In [3]:
REQ = ("document_id", "utterance_id", "input", "target", "split")
KEEP_FLAGS = ("change_type", "flag_digit_changed", "flag_alpha_changed", "flag_emoji_changed")
D = {}                      # split -> dict of lists
COLS_SEEN = {}; MISSING_KEY_ROWS = {}
t0 = time.time()
for s, p in FILES.items():
    d = {k: [] for k in ("document_id", "utterance_id", "input", "target", "split_field", *KEEP_FLAGS)}
    miss = 0; cols = None
    with open(p, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            r = json.loads(line)
            if cols is None:
                cols = list(r.keys())
            if any(k not in r for k in REQ):
                miss += 1
            d["document_id"].append(r.get("document_id")); d["utterance_id"].append(r.get("utterance_id"))
            d["input"].append(r.get("input")); d["target"].append(r.get("target")); d["split_field"].append(r.get("split"))
            for k in KEEP_FLAGS:
                d[k].append(r.get(k))
    D[s] = d; COLS_SEEN[s] = cols; MISSING_KEY_ROWS[s] = miss
    print(f"{s:10s} {len(d['input']):>9,d}행 로드 ({time.time()-t0:.0f}초 경과)")
print("\ncolumn 수:", {s: len(c) for s, c in COLS_SEEN.items()}, "| 세 파일 column 동일:", COLS_SEEN["train"] == COLS_SEEN["validation"] == COLS_SEEN["test"])
print("column 목록(train):", COLS_SEEN["train"])

train        986,718행 로드 (7초 경과)
validation    54,730행 로드 (7초 경과)
test          54,954행 로드 (8초 경과)

column 수: {'train': 38, 'validation': 38, 'test': 38} | 세 파일 column 동일: True
column 목록(train): ['document_id', 'utterance_id', 'speaker_id', 'utterance_index', 'split', 'publisher', 'topic', 'relation', 'speaker_sex', 'speaker_age', 'input', 'target', 'change_type', 'input_len', 'target_len', 'len_ratio', 'edit_ratio', 'punct_added', 'train_n_variants', 'flag_form_conflict', 'flag_form_conflict_long', 'flag_edit_large', 'flag_digit_changed', 'flag_alpha_changed', 'flag_emoji_changed', 'flag_punct_excess', 'flag_missed_correction_suspect', 'flag_partial_correction_suspect', 'flag_len_shrink', 'flag_deid_case_only', 'flag_input_has_linebreak', 'flag_seen_input_in_train', 'flag_seen_pair_in_train', 'n_quality_flags', 'quality_flags', 'form_raw', 'corrected_form_raw', 'original_form']


## [셀 4] A. 기본 구조
행 수, 필수 column, split 필드, document 수를 지수의 기록(`FINAL_MANIFEST.json`, `final_summary.csv`, `document_split_map.csv`)과 비교한다.

In [4]:
N = {s: len(D[s]["input"]) for s in D}
for s in D:
    check("STOP", f"행 수 {s} = manifest counts_final", N[s] == MANIFEST["counts_final"][s], f"실제 {N[s]:,} / 기록 {MANIFEST['counts_final'][s]:,}")
    check("STOP", f"필수 column 존재 {s}", all(k in COLS_SEEN[s] for k in REQ) and MISSING_KEY_ROWS[s] == 0, f"필수 키 누락 행 {MISSING_KEY_ROWS[s]}")
    bad_split = sum(1 for x in D[s]["split_field"] if x != s)
    check("STOP", f"split 필드 = 파일명 {s}", bad_split == 0, f"불일치 {bad_split}행")

SUMMARY = pd.read_csv(EXTRA["summary"], encoding="utf-8-sig", index_col=0)
DOCS = {s: len(set(D[s]["document_id"])) for s in D}
for s in D:
    rec = int(SUMMARY.loc[s, "문서 수"])
    check("STOP", f"document 수 {s} = final_summary.csv", DOCS[s] == rec, f"실제 {DOCS[s]:,} / 기록 {rec:,}")
    check("STOP", f"최종 행 수 {s} = final_summary.csv", N[s] == int(SUMMARY.loc[s, "최종 행 수"]), f"실제 {N[s]:,}")
removed_rows = [json.loads(l) for l in open(EXTRA["removed"], encoding="utf-8") if l.strip()]
check("STOP", "제거 행 수 = train 제거 기록(10)", len(removed_rows) == int(SUMMARY.loc["train", "제거"]) == MANIFEST["counts_before"]["train"] - MANIFEST["counts_final"]["train"],
      f"removed_rows.jsonl {len(removed_rows)}행, manifest 차이 {MANIFEST['counts_before']['train'] - MANIFEST['counts_final']['train']}")
train_uid = set(D["train"]["utterance_id"])
still_there = [r["utterance_id"] for r in removed_rows if r["utterance_id"] in train_uid]
check("STOP", "제거 기록된 utterance_id가 train에 남아 있지 않음", not still_there, f"잔존 {len(still_there)}건")
pd.DataFrame({"split": list(N), "rows": [N[s] for s in N], "documents": [DOCS[s] for s in N], "rows_per_doc": [round(N[s] / DOCS[s], 1) for s in N]})

통과 [STOP] 행 수 train = manifest counts_final — 실제 986,718 / 기록 986,718
통과 [STOP] 필수 column 존재 train — 필수 키 누락 행 0
통과 [STOP] split 필드 = 파일명 train — 불일치 0행
통과 [STOP] 행 수 validation = manifest counts_final — 실제 54,730 / 기록 54,730
통과 [STOP] 필수 column 존재 validation — 필수 키 누락 행 0
통과 [STOP] split 필드 = 파일명 validation — 불일치 0행
통과 [STOP] 행 수 test = manifest counts_final — 실제 54,954 / 기록 54,954
통과 [STOP] 필수 column 존재 test — 필수 키 누락 행 0
통과 [STOP] split 필드 = 파일명 test — 불일치 0행
통과 [STOP] document 수 train = final_summary.csv — 실제 18,328 / 기록 18,328
통과 [STOP] 최종 행 수 train = final_summary.csv — 실제 986,718
통과 [STOP] document 수 validation = final_summary.csv — 실제 1,527 / 기록 1,527
통과 [STOP] 최종 행 수 validation = final_summary.csv — 실제 54,730
통과 [STOP] document 수 test = final_summary.csv — 실제 1,465 / 기록 1,465
통과 [STOP] 최종 행 수 test = final_summary.csv — 실제 54,954
통과 [STOP] 제거 행 수 = train 제거 기록(10) — removed_rows.jsonl 10행, manifest 차이 10
통과 [STOP] 제거 기록된 utterance_id가 train에 남아 있지 않음 — 잔존 0건


        split    rows  documents  rows_per_doc
0       train  986718      18328          53.8
1  validation   54730       1527          35.8
2        test   54954       1465          37.5

## [셀 5] B. 결측 / 빈 문장
null, 문자열이 아닌 값, 공백뿐인 문자열을 센다. (지수 전처리 정규화는 `NFC + 앞뒤 공백 제거`뿐이므로, 이 검증도 그 기준 그대로 본다.)

In [5]:
rows = []
for s in D:
    inp, tgt = D[s]["input"], D[s]["target"]
    ni = sum(x is None for x in inp); nt = sum(x is None for x in tgt)
    nsi = sum((x is not None) and (not isinstance(x, str)) for x in inp); nst = sum((x is not None) and (not isinstance(x, str)) for x in tgt)
    bi = sum(isinstance(x, str) and x.strip() == "" for x in inp); bt = sum(isinstance(x, str) and x.strip() == "" for x in tgt)
    ws_i = sum(isinstance(x, str) and x != x.strip() for x in inp); ws_t = sum(isinstance(x, str) and x != x.strip() for x in tgt)
    nfc_bad = sum(isinstance(x, str) and not unicodedata.is_normalized("NFC", x) for x in inp) + sum(isinstance(x, str) and not unicodedata.is_normalized("NFC", x) for x in tgt)
    rows.append({"split": s, "input null": ni, "target null": nt, "input 비문자열": nsi, "target 비문자열": nst, "input 공백/빈 문자열": bi, "target 공백/빈 문자열": bt,
                 "앞뒤 공백 남은 input": ws_i, "앞뒤 공백 남은 target": ws_t, "NFC 아닌 문자열(input+target)": nfc_bad})
    check("STOP", f"결측·빈 문장 없음 {s}", ni + nt + nsi + nst + bi + bt == 0, f"input null {ni}, target null {nt}, 빈/공백 input {bi}, 빈/공백 target {bt}")
    check("WARN", f"정규화(NFC+strip) 상태 유지 {s}", ws_i + ws_t + nfc_bad == 0, f"앞뒤 공백 {ws_i + ws_t}, NFC 아님 {nfc_bad}")
pd.DataFrame(rows).set_index("split").T

통과 [STOP] 결측·빈 문장 없음 train — input null 0, target null 0, 빈/공백 input 0, 빈/공백 target 0
통과 [WARN] 정규화(NFC+strip) 상태 유지 train — 앞뒤 공백 0, NFC 아님 0
통과 [STOP] 결측·빈 문장 없음 validation — input null 0, target null 0, 빈/공백 input 0, 빈/공백 target 0
통과 [WARN] 정규화(NFC+strip) 상태 유지 validation — 앞뒤 공백 0, NFC 아님 0
통과 [STOP] 결측·빈 문장 없음 test — input null 0, target null 0, 빈/공백 input 0, 빈/공백 target 0
통과 [WARN] 정규화(NFC+strip) 상태 유지 test — 앞뒤 공백 0, NFC 아님 0


split                     train  validation  test
input null                    0           0     0
target null                   0           0     0
input 비문자열                    0           0     0
target 비문자열                   0           0     0
input 공백/빈 문자열                0           0     0
target 공백/빈 문자열               0           0     0
앞뒤 공백 남은 input                0           0     0
앞뒤 공백 남은 target               0           0     0
NFC 아닌 문자열(input+target)      0           0     0

## [셀 6] C. 중복
- `utterance_id` 중복은 STOP 대상이다.
- 동일 input / target / input-target pair 중복은 **삭제 대상이 아니라 현황 파악 대상**이다(짧은 대화 “네”, “ㅋㅋ”는 자연스럽게 반복된다).
- split 간 겹침은 Train에 있는 문장이 Validation/Test에도 있는지 보는 것이므로, 겹침 자체보다 **어떤 길이의 문장이 겹치는지**가 중요하다. 지수 전처리는 긴 문장을 공유하는 문서를 같은 split에 묶도록 설계했으므로 그 설계가 지켜졌는지도 확인한다.

In [6]:
# utterance_id
for s in D:
    u = D[s]["utterance_id"]
    check("STOP", f"utterance_id 중복 없음 {s}", len(u) == len(set(u)), f"중복 {len(u) - len(set(u))}건")
U = {s: set(D[s]["utterance_id"]) for s in D}
for a, b in (("train", "validation"), ("train", "test"), ("validation", "test")):
    check("STOP", f"utterance_id split 간 겹침 없음 {a}∩{b}", len(U[a] & U[b]) == 0, f"{len(U[a] & U[b])}건")

# split 내부 동일 input / target / pair
rows = []
for s in D:
    inp, tgt = D[s]["input"], D[s]["target"]
    pairs = list(zip(inp, tgt))
    rows.append({"split": s, "행": len(inp),
                 "동일 input 중복 행": len(inp) - len(set(inp)), "동일 target 중복 행": len(tgt) - len(set(tgt)), "동일 pair 중복 행": len(pairs) - len(set(pairs))})
WITHIN = pd.DataFrame(rows).set_index("split"); print("[split 내부 중복 행 수(첫 등장 제외)]"); print(WITHIN.to_string())

통과 [STOP] utterance_id 중복 없음 train — 중복 0건
통과 [STOP] utterance_id 중복 없음 validation — 중복 0건
통과 [STOP] utterance_id 중복 없음 test — 중복 0건
통과 [STOP] utterance_id split 간 겹침 없음 train∩validation — 0건
통과 [STOP] utterance_id split 간 겹침 없음 train∩test — 0건
통과 [STOP] utterance_id split 간 겹침 없음 validation∩test — 0건
[split 내부 중복 행 수(첫 등장 제외)]
                 행  동일 input 중복 행  동일 target 중복 행  동일 pair 중복 행
split                                                          
train       986718         157738          176192        146737
validation   54730           5279            5861          4814
test         54954           4716            5416          4289


In [7]:
# split 간 동일 input / 동일 pair, 그리고 '긴 문장' 기준(근접중복 그룹 설계 확인)
def key_ns(x): return re.sub(r"\s+", "", x)
def hangul_n(x): return len(re.findall(r"[가-힣]", x))
DUP_MIN_LEN, DUP_MIN_HANGUL = 15, 5          # 지수 preprocess_config.json thresholds와 동일해야 함
cfg_thr = json.loads(EXTRA["config"].read_text(encoding="utf-8"))["thresholds"]
assert (cfg_thr["DUP_MIN_LEN"], cfg_thr["DUP_MIN_HANGUL"]) == (DUP_MIN_LEN, DUP_MIN_HANGUL), "지수 설정과 임계값이 다릅니다"

tr_inputs = set(D["train"]["input"]); tr_pairs = set(zip(D["train"]["input"], D["train"]["target"]))
tr_long_keys = {key_ns(x) for x in D["train"]["input"] if len(key_ns(x)) >= DUP_MIN_LEN and hangul_n(x) >= DUP_MIN_HANGUL}
rows = []; LONG_OVERLAP_EXAMPLES = {}
for s in ("validation", "test"):
    inp, tgt = D[s]["input"], D[s]["target"]
    same_in = [i for i, x in enumerate(inp) if x in tr_inputs]
    same_pair = [i for i, (x, y) in enumerate(zip(inp, tgt)) if (x, y) in tr_pairs]
    long_ov = [i for i, x in enumerate(inp) if len(key_ns(x)) >= DUP_MIN_LEN and hangul_n(x) >= DUP_MIN_HANGUL and key_ns(x) in tr_long_keys]
    LONG_OVERLAP_EXAMPLES[s] = long_ov
    rows.append({"split": s, "행": len(inp), "Train과 동일 input 행": len(same_in), "비율": len(same_in) / len(inp),
                 "Train과 동일 pair 행": len(same_pair), "pair 비율": len(same_pair) / len(inp),
                 "긴 문장(≥15자·한글≥5) Train과 동일 행": len(long_ov), "긴 문장 비율": len(long_ov) / len(inp)})
    check("STOP", f"긴 문장 근접중복 그룹 설계 준수 {s}", len(long_ov) / len(inp) <= 0.001, f"{len(long_ov)}행 ({len(long_ov)/len(inp):.4%}), 기준 0.1% 이하")
    check("WARN", f"짧은/일반 문장 포함 Train과 동일 input {s}", len(same_in) == 0, f"{len(same_in):,}행 ({len(same_in)/len(inp):.2%}) — 삭제하지 않고 기록")
CROSS = pd.DataFrame(rows).set_index("split")
print(CROSS.to_string(float_format=lambda v: f"{v:.4f}"))
# 겹치는 input의 길이 분포(어떤 문장이 겹치는가)
for s in ("validation", "test"):
    lens = [len(key_ns(x)) for x in D[s]["input"] if x in tr_inputs]
    if lens:
        q = np.quantile(lens, [0.5, 0.9, 0.99]); print(f"[{s}] Train과 겹치는 input의 공백 제외 길이 중앙값/90%/99%: {q[0]:.0f} / {q[1]:.0f} / {q[2]:.0f}  (max {max(lens)})")
print("\n[Validation 겹침 예시 — Validation만 출력]")
ex = [i for i, x in enumerate(D["validation"]["input"]) if x in tr_inputs]
random.Random(SEED).shuffle(ex)
for i in ex[:8]: print("  ", repr(D["validation"]["input"][i]), "→", repr(D["validation"]["target"][i]))

통과 [STOP] 긴 문장 근접중복 그룹 설계 준수 validation — 0행 (0.0000%), 기준 0.1% 이하
주의 [WARN] 짧은/일반 문장 포함 Train과 동일 input validation — 9,301행 (16.99%) — 삭제하지 않고 기록
통과 [STOP] 긴 문장 근접중복 그룹 설계 준수 test — 0행 (0.0000%), 기준 0.1% 이하
주의 [WARN] 짧은/일반 문장 포함 Train과 동일 input test — 8,501행 (15.47%) — 삭제하지 않고 기록
                행  Train과 동일 input 행     비율  Train과 동일 pair 행  pair 비율  긴 문장(≥15자·한글≥5) Train과 동일 행  긴 문장 비율
split                                                                                                       
validation  54730               9301 0.1699              8678   0.1586                            0   0.0000
test        54954               8501 0.1547              7924   0.1442                            0   0.0000
[validation] Train과 겹치는 input의 공백 제외 길이 중앙값/90%/99%: 3 / 8 / 20  (max 74)
[test] Train과 겹치는 input의 공백 제외 길이 중앙값/90%/99%: 3 / 8 / 21  (max 56)

[Validation 겹침 예시 — Validation만 출력]
   '글치글치' → '글치글치.'
   '대박!!!!!' → '대박!'
   '고양이는' → '고양이는'
   '등등등' → '등등등.'
   '어 웅웅 알았어요' → '어, 웅웅, 

## [셀 7] D. 데이터 누수 (document_id)
`Train ∩ Validation`, `Train ∩ Test`, `Validation ∩ Test`의 `document_id` 교집합이 모두 0이어야 한다. 교집합이 하나라도 있으면 STOP이며 학습을 시작하지 않는다. 추가로 `document_split_map.csv`의 문서 배정과 실제 파일의 배정이 같은지 확인한다.

In [8]:
DS = {s: set(D[s]["document_id"]) for s in D}
INTER = {}
for a, b in (("train", "validation"), ("train", "test"), ("validation", "test")):
    INTER[f"{a}∩{b}"] = len(DS[a] & DS[b])
    check("STOP", f"document_id 교집합 = 0 ({a}∩{b})", len(DS[a] & DS[b]) == 0, f"교집합 {len(DS[a] & DS[b])}개")
print("document 수:", {s: len(DS[s]) for s in DS}, "| 합계", sum(len(v) for v in DS.values()))

smap = pd.read_csv(EXTRA["split_map"], encoding="utf-8-sig")
map_assign = dict(zip(smap["document_id"], smap["split"]))
file_assign = {doc: s for s in DS for doc in DS[s]}
diff = [d for d, s in file_assign.items() if map_assign.get(d) != s]
not_in_files = [d for d in map_assign if d not in file_assign]
check("STOP", "파일 속 문서의 split 배정 = document_split_map.csv", len(diff) == 0, f"배정 불일치 문서 {len(diff)}개")
check("WARN", "document_split_map에만 있고 파일에 없는 문서", len(not_in_files) == 0, f"{len(not_in_files)}개 (전량 제거·사용 불가 행 문서일 수 있음)")
print("map 문서 수:", len(map_assign), "| 파일에 등장한 문서:", len(file_assign))
# 그룹 단위(같은 긴 문장을 공유하는 문서 묶음)가 한 split에 모여 있는지
grp = dict(zip(smap["document_id"], smap["group_id"]))
grp_splits = defaultdict(set)
for d, s in file_assign.items(): grp_splits[grp.get(d, d)].add(s)
mixed_groups = sum(1 for v in grp_splits.values() if len(v) > 1)
check("STOP", "group_id 하나가 둘 이상의 split에 걸치지 않음", mixed_groups == 0, f"여러 split에 걸친 group {mixed_groups}개 / 전체 {len(grp_splits):,}")
share = {s: len(DS[s]) / sum(len(v) for v in DS.values()) for s in DS}; thr = json.loads(EXTRA["config"].read_text(encoding="utf-8"))["split_ratios"]
print("document 비율(실제) vs 설정:", {s: (round(share[s], 4), thr[s]) for s in DS})

통과 [STOP] document_id 교집합 = 0 (train∩validation) — 교집합 0개
통과 [STOP] document_id 교집합 = 0 (train∩test) — 교집합 0개
통과 [STOP] document_id 교집합 = 0 (validation∩test) — 교집합 0개
document 수: {'train': 18328, 'validation': 1527, 'test': 1465} | 합계 21320
통과 [STOP] 파일 속 문서의 split 배정 = document_split_map.csv — 배정 불일치 문서 0개
주의 [WARN] document_split_map에만 있고 파일에 없는 문서 — 15개 (전량 제거·사용 불가 행 문서일 수 있음)
map 문서 수: 21335 | 파일에 등장한 문서: 21320
통과 [STOP] group_id 하나가 둘 이상의 split에 걸치지 않음 — 여러 split에 걸친 group 0개 / 전체 21,161
document 비율(실제) vs 설정: {'train': (0.8597, 0.9), 'validation': (0.0716, 0.05), 'test': (0.0687, 0.05)}


## [셀 8] E. 길이 / 이상치 (후보만 출력, 자동 삭제 없음)
길이는 문자 수 기준이다. 아래 후보는 **삭제 대상이 아니라 사람이 볼 후보**다.

In [9]:
def length_stats(lst):
    a = np.array([len(x) for x in lst]); q = np.quantile(a, [0.01, 0.5, 0.9, 0.99, 0.999])
    return {"min": int(a.min()), "1%": q[0], "중앙값": q[1], "90%": q[2], "99%": q[3], "99.9%": q[4], "max": int(a.max()), "평균": float(a.mean())}
LEN_TAB = pd.DataFrame({f"{s}_{c}": length_stats(D[s][c]) for s in D for c in ("input", "target")}).T
print(LEN_TAB.round(1).to_string())

CAND = defaultdict(dict); CAND_IDX = {}
for s in D:
    il = np.array([len(x) for x in D[s]["input"]]); tl = np.array([len(x) for x in D[s]["target"]])
    ratio = tl / np.maximum(il, 1)
    masks = {"input 1글자": il == 1, "input ≥300자": il >= 300, "target ≥300자": tl >= 300,
             "target/input > 3 (input≥5)": (ratio > 3) & (il >= 5), "target/input < 0.3 (input≥5)": (ratio < 0.3) & (il >= 5)}
    for k, m in masks.items():
        CAND[s][k] = int(m.sum()); CAND_IDX[(s, k)] = np.where(m)[0]
CAND_TAB = pd.DataFrame(CAND); CAND_TAB.loc["(비율%) target/input>3"] = [100 * CAND[s]["target/input > 3 (input≥5)"] / len(D[s]["input"]) for s in D]
CAND_TAB.loc["(비율%) target/input<0.3"] = [100 * CAND[s]["target/input < 0.3 (input≥5)"] / len(D[s]["input"]) for s in D]
print("\n[이상치 후보 수]"); print(CAND_TAB.round(4).to_string())
check("WARN", "target이 input보다 지나치게 긴 후보(>3배)", CAND["train"]["target/input > 3 (input≥5)"] == 0, f"train {CAND['train']['target/input > 3 (input≥5)']}행, val {CAND['validation']['target/input > 3 (input≥5)']}행")
check("WARN", "target이 지나치게 짧아진 후보(<0.3배)", CAND["train"]["target/input < 0.3 (input≥5)"] == 0, f"train {CAND['train']['target/input < 0.3 (input≥5)']}행, val {CAND['validation']['target/input < 0.3 (input≥5)']}행")
for k in ("target/input > 3 (input≥5)", "target/input < 0.3 (input≥5)"):
    idx = list(CAND_IDX[("validation", k)]); random.Random(SEED).shuffle(idx)
    print(f"\n[Validation 후보 예시: {k}] ({len(idx)}행 중 최대 6개)")
    for i in idx[:6]: print("  ", repr(D["validation"]["input"][i][:80]), "→", repr(D["validation"]["target"][i][:80]))

                   min   1%   중앙값   90%   99%  99.9%     max    평균
train_input        1.0  1.0  11.0  26.0  52.0   90.0  6916.0  13.3
train_target       1.0  2.0  12.0  28.0  56.0   96.0  6924.0  14.9
validation_input   1.0  1.0  11.0  26.0  53.0   97.0   717.0  13.6
validation_target  1.0  2.0  13.0  29.0  57.0  100.0   679.0  15.2
test_input         1.0  1.0  12.0  27.0  54.0   94.0   501.0  14.2
test_target        1.0  2.0  13.0  30.0  58.0   98.0   482.0  15.8

[이상치 후보 수]
                                   train  validation       test
input 1글자                     24617.0000   1179.0000  1033.0000
input ≥300자                      96.0000      4.0000     2.0000
target ≥300자                     91.0000      5.0000     2.0000
target/input > 3 (input≥5)        0.0000      0.0000     0.0000
target/input < 0.3 (input≥5)    680.0000     36.0000    47.0000
(비율%) target/input>3              0.0000      0.0000     0.0000
(비율%) target/input<0.3            0.0689      0.0658     0.0855
통과 [WAR

## [셀 9] F. 주요 내용 보존 후보
숫자·영어·URL·이메일·자모·특수문자가 input→target에서 바뀐 행을 **독립적으로** 다시 계산한다. 지수의 `flag_digit_changed`, `flag_alpha_changed`와 비교해 두 계산이 얼마나 일치하는지도 본다. 일부 변화(예: “ㅋㅋ”→“ㅋㅋ.”)는 정상 교정일 수 있으므로 결과는 오류 확정이 아니라 후보 수다.

In [10]:
RE_DIG = re.compile(r"\d+"); RE_ENG = re.compile(r"[A-Za-z]+"); RE_URL = re.compile(r"https?://\S+|www\.\S+"); RE_MAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
RE_JAMO = re.compile(r"[ㄱ-ㅎㅏ-ㅣ]+"); RE_SPECIAL = re.compile(r"[^\w\s가-힣ㄱ-ㅎㅏ-ㅣ.,!?~…]")
def ms(rx, x): return Counter(rx.findall(x))
FEATS = {"숫자 변경": RE_DIG, "영어 변경": RE_ENG, "자모(ㅋㅋ 등) 변경": RE_JAMO, "특수문자 변경": RE_SPECIAL}
rows = []; FLAG_IDX = {}
for s in D:
    inp, tgt = D[s]["input"], D[s]["target"]; row = {"split": s}
    for name, rx in FEATS.items():
        idx = [i for i in range(len(inp)) if ms(rx, inp[i]) != ms(rx, tgt[i])]
        row[name] = len(idx); FLAG_IDX[(s, name)] = idx
    for name, rx in (("URL 삭제/변경", RE_URL), ("이메일 삭제/변경", RE_MAIL)):
        idx = [i for i in range(len(inp)) if ms(rx, inp[i]) and ms(rx, inp[i]) != ms(rx, tgt[i])]
        row[name] = len(idx); FLAG_IDX[(s, name)] = idx
    rows.append(row)
FEAT_TAB = pd.DataFrame(rows).set_index("split"); print("[내용 보존 후보 수(행)]"); print(FEAT_TAB.to_string())
print("\n[전체 대비 %]"); print((FEAT_TAB.div([len(D[s]['input']) for s in FEAT_TAB.index], axis=0) * 100).round(3).to_string())

# 지수 플래그와의 일치도(Train, Validation)
print("\n[독립 계산 vs 지수 플래그 — 교차표]")
AGREE = {}
for s in ("train", "validation"):
    for name, flag in (("숫자 변경", "flag_digit_changed"), ("영어 변경", "flag_alpha_changed")):
        mine = np.zeros(len(D[s]["input"]), bool); mine[FLAG_IDX[(s, name)]] = True
        theirs = np.array([bool(x) for x in D[s][flag]])
        AGREE[(s, name)] = {"둘 다": int((mine & theirs).sum()), "나만": int((mine & ~theirs).sum()), "지수만": int((~mine & theirs).sum())}
        print(f"  {s:10s} {name}: {AGREE[(s, name)]}")
check("WARN", "URL 삭제/변경 후보 없음(train)", FEAT_TAB.loc["train", "URL 삭제/변경"] == 0, f"{FEAT_TAB.loc['train', 'URL 삭제/변경']}행")
check("WARN", "이메일 삭제/변경 후보 없음(train)", FEAT_TAB.loc["train", "이메일 삭제/변경"] == 0, f"{FEAT_TAB.loc['train', '이메일 삭제/변경']}행")
for name in ("숫자 변경", "영어 변경"):
    idx = list(FLAG_IDX[("validation", name)]); random.Random(SEED).shuffle(idx)
    print(f"\n[Validation 예시: {name}] ({len(idx)}행 중 6개)")
    for i in idx[:6]: print("  ", repr(D["validation"]["input"][i][:70]), "→", repr(D["validation"]["target"][i][:70]))

[내용 보존 후보 수(행)]
            숫자 변경  영어 변경  자모(ㅋㅋ 등) 변경  특수문자 변경  URL 삭제/변경  이메일 삭제/변경
split                                                               
train        1369    429        35352    10980          0          0
validation     74     19         1919      633          0          0
test           86     20         2053      653          0          0

[전체 대비 %]
            숫자 변경  영어 변경  자모(ㅋㅋ 등) 변경  특수문자 변경  URL 삭제/변경  이메일 삭제/변경
split                                                               
train       0.139  0.043        3.583    1.113        0.0        0.0
validation  0.135  0.035        3.506    1.157        0.0        0.0
test        0.156  0.036        3.736    1.188        0.0        0.0

[독립 계산 vs 지수 플래그 — 교차표]
  train      숫자 변경: {'둘 다': 1369, '나만': 0, '지수만': 0}
  train      영어 변경: {'둘 다': 232, '나만': 197, '지수만': 0}
  validation 숫자 변경: {'둘 다': 74, '나만': 0, '지수만': 0}
  validation 영어 변경: {'둘 다': 12, '나만': 7, '지수만': 0}
통과 [WARN] URL 삭제/변경 후보 없음(train) — 0행
통과 [WARN] 이

## [셀 10] G. 기존 기록과의 일치 + change_type 분포
지수 기록 (`FINAL_MANIFEST.json`, `final_summary.csv`)과 실제 파일의 일치는 셀 2·4에서 이미 확인했다. 여기서는 `change_type` 분포를 Train/Validation/Test에서 비교해 세 split의 성격이 크게 다르지 않은지 본다.

In [11]:
CT = pd.DataFrame({s: pd.Series(D[s]["change_type"]).value_counts(normalize=True) for s in D}).fillna(0)
print((CT * 100).round(2).to_string())
same_ratio = {s: float(np.mean([a == b for a, b in zip(D[s]["input"], D[s]["target"])])) for s in D}
print("\ninput == target 비율(교정 불필요 문장):", {s: round(v, 4) for s, v in same_ratio.items()})
tvd = 0.5 * (CT["train"] - CT["validation"]).abs().sum(), 0.5 * (CT["train"] - CT["test"]).abs().sum()
check("WARN", "change_type 분포 Train vs Validation/Test 차이(총변동거리)가 작음(<0.05)", max(tvd) < 0.05, f"Train-Validation {tvd[0]:.4f}, Train-Test {tvd[1]:.4f}")

                  train  validation   test
punct_only        53.29       51.85  52.55
spelling_or_word  24.10       25.83  25.89
unchanged         13.86       13.46  12.83
spacing_only       8.75        8.86   8.73

input == target 비율(교정 불필요 문장): {'train': 0.1386, 'validation': 0.1346, 'test': 0.1283}
통과 [WARN] change_type 분포 Train vs Validation/Test 차이(총변동거리)가 작음(<0.05) — Train-Validation 0.0184, Train-Test 0.0179


## [셀 11] H. 검증 판정 (PASS / STOP)
STOP 등급 항목이 하나라도 실패하면 **STOP**이다. 이 경우 지수 데이터를 자동 수정하지 않고, 아래 다섯 가지를 보고한 뒤 사용자가 수정 여부를 결정한다.
1. 어떤 문제가 발견되었는가  2. 어느 데이터에서 발생했는가  3. 모델 학습에 어떤 영향을 주는가  4. 지수 원본을 고쳐야 하는 문제인가  5. adapter나 별도 검증 코드로 해결 가능한가

In [12]:
RES = pd.DataFrame(CHECKS)
stop_fail = RES[(RES["level"] == "STOP") & (~RES["pass"])]
warns = RES[(RES["level"] == "WARN") & (~RES["pass"])]
VERDICT = "PASS" if stop_fail.empty else "STOP"
print("STOP 등급 검사:", int((RES["level"] == "STOP").sum()), "개 | 실패", len(stop_fail), "개")
print("WARN 등급 검사:", int((RES["level"] == "WARN").sum()), "개 | 주의", len(warns), "개")
if not stop_fail.empty:
    print("\n[STOP 사유]"); print(stop_fail[["check", "detail"]].to_string(index=False))
if not warns.empty:
    print("\n[WARN 목록 — 최종 보고서에 기록]"); print(warns[["check", "detail"]].to_string(index=False))
print("\n" + "=" * 60); print("검증 판정:", VERDICT); print("=" * 60)

RESULT = {"verdict": VERDICT, "verified_at": time.strftime("%Y-%m-%d %H:%M:%S"), "dataset_version": MANIFEST["dataset_version"],
          "data_dir": str(DATA_DIR), "python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__,
          "sha256_actual": ACTUAL_SHA, "rows": N, "documents": DOCS, "document_intersections": INTER,
          "stop_failures": stop_fail[["check", "detail"]].to_dict("records"), "warnings": warns[["check", "detail"]].to_dict("records"),
          "all_checks": CHECKS, "data_modified": False, "test_used_for": "구조 검증(행 수·누수·중복·길이·내용 보존 후보 수)만. 성능 평가·모델 선택 미사용.",
          "cross_split_overlap": CROSS.reset_index().to_dict("records"), "length_candidates": CAND_TAB.to_dict(), "content_preservation_candidates": FEAT_TAB.to_dict()}
(OUT_DIR / "verification_result.json").write_text(json.dumps(RESULT, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
LEN_TAB.round(2).to_csv(OUT_DIR / "length_stats.csv", encoding="utf-8-sig"); RES.to_csv(OUT_DIR / "checks.csv", index=False, encoding="utf-8-sig")
print("저장:", OUT_DIR)
assert VERDICT == "PASS", "STOP — 06번 이후 모델링으로 진행하지 않습니다. 위 사유를 보고하세요."

STOP 등급 검사: 36 개 | 실패 0 개
WARN 등급 검사: 11 개 | 주의 4 개

[WARN 목록 — 최종 보고서에 기록]
                                 check                       detail
짧은/일반 문장 포함 Train과 동일 input validation 9,301행 (16.99%) — 삭제하지 않고 기록
      짧은/일반 문장 포함 Train과 동일 input test 8,501행 (15.47%) — 삭제하지 않고 기록
     document_split_map에만 있고 파일에 없는 문서 15개 (전량 제거·사용 불가 행 문서일 수 있음)
            target이 지나치게 짧아진 후보(<0.3배)          train 680행, val 36행

검증 판정: PASS
저장: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/integrated_korean_correction/data/processed/verification_05


## 해석 (위 실행 결과를 확인한 뒤 작성)

**판정: PASS.** STOP 등급 36개 검사를 모두 통과했다. 지수의 `preprocessed_final_v1`은 SHA256(3개 파일), 행 수(986,718 / 54,730 / 54,954), 문서 수(18,328 / 1,527 / 1,465), 필수 column, 결측·빈 문자열(0건), `utterance_id` 중복(0건), `document_id` split 간 교집합(0개), `document_split_map.csv`와의 배정 일치, 그룹 단위 split 분리(21,161개 group 중 여러 split에 걸친 것 0개)가 모두 기록과 일치했다. 검증 중 데이터는 수정하지 않았다. → 모델링(06~)으로 진행할 수 있다.

**모델링 결과를 해석할 때 반드시 기억할 주의사항(WARN 4건, 삭제하지 않고 기록):**
1. **짧은 문장이 Train과 겹친다.** Validation의 16.99%(9,301행), Test의 15.47%(8,501행)가 Train에 똑같은 input이 있다. 겹치는 input의 공백 제외 길이 중앙값은 3자(99%는 20~21자)로 “ㅋㅋ”, “네?”, “대박!!!!!” 같은 짧은 대화다. 긴 문장(15자 이상)의 겹침은 0행이어서 지수의 근접중복 그룹 설계는 지켜졌다. 이 겹침 때문에 짧은 문장에서는 모델이 정답을 외워서 맞힐 수 있으므로, 이후 평가에서 **겹치는 행/안 겹치는 행을 나누어 보고**하는 것이 필요하다(11번 노트북에서 수행).
2. **target이 input의 0.3배 미만으로 짧아진 행**이 Train 680행(0.07%), Validation 36행이다. 예시는 “음.........................” → “음...”, “???????” → “?”처럼 반복 문장부호 축약이라 대부분 정상 교정으로 보이지만, 전수 확인은 하지 않았다(후보만 출력).
3. **document 비율**이 설정(90/5/5)과 다르다(문서 기준 86.0 / 7.2 / 6.9%). 지수는 같은 긴 문장을 공유하는 문서를 그룹으로 묶어 같은 split에 배정했고, 행 수 기준으로는 Validation·Test가 각각 약 5%다. 이 차이가 누수는 아니다.
4. **document_split_map에만 있고 파일에 없는 문서 15개.** 사용 가능한 행이 없어 최종 데이터에서 빠진 문서로 추정되지만, 원인은 이 노트북에서 확인하지 않았다(확인 불가).

**라벨 품질 관련 관찰(이 노트북의 판정 대상은 아님, 라벨 감사 노트북의 입력으로 사용):**
- 내용 보존 후보에서 target 자체가 의심스러운 예가 있다: “ㄷㄷㄷ” → “eee”, “놓치면 ㄷㄷ” → “놓치면 ee”(자판 변환 결과가 그대로 남은 것으로 보임), “안녕하세요 name1” → “안녕하세요? Name1.”(비식별 토큰의 대문자화). 독립 계산의 ‘영어 변경’ Train 429행 중 197행은 지수 플래그(`flag_alpha_changed`)에는 잡히지 않았는데, 지수가 비식별 토큰 대소문자 변화를 별도 플래그(`flag_deid_case_only`)로 다루기 때문일 가능성이 있으나 이 노트북에서 확인하지는 않았다.
- 숫자 변경 후보 1,369행(Train)은 지수 플래그와 완전히 일치했다(‘나만’ 0, ‘지수만’ 0).

**한계:** 이 검증은 형식·구조·기록 일치·누수 위주다. target(정답)의 언어적 정확성은 검증하지 않았다.